# 04 分区建模 + 空间交叉验证 + 模型对比（任务书核心）

In [1]:

import sys
from pathlib import Path
# 让 notebook 能 import src
sys.path.append(str(Path('..').resolve()))


import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir, set_seed

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
FIG = ensure_dir(OUT/'figures')
df = pd.read_parquet(OUT/'clean_data.parquet')


from pathlib import Path
from src.spatial_cv import make_spatial_groups, get_group_kfold
from src.models import fit_oof_with_spatialcv
from src.utils import safe_col, ensure_dir
from src.plots import obs_pred_scatter, residual_plot

# --- 输出目录（关键：避免 FIG 未定义）---
OUT = ensure_dir('../outputs')  # 若你前面已定义 OUT，这行可删
FIG = ensure_dir(OUT / "figures")

# --- 数据准备 ---
bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])
max_rows = int(cfg['modeling']['max_train_rows'])
df_run = (
    df.sample(max_rows, random_state=cfg['project']['seed']).reset_index(drop=True)
    if len(df) > max_rows else df.reset_index(drop=True)
)

cv = get_group_kfold(cfg)

group_keys = ['crop', 'ph_bin'] if 'crop' in df_run.columns else ['ph_bin']
models = cfg['modeling']['models']
rows = []

def safe_key(x):
    # 更稳的文件名（避免空格、括号、逗号等）
    return str(x).replace(" ", "").replace("(", "").replace(")", "").replace(",", "_").replace("'", "")

for key, sub in df_run.groupby(group_keys, dropna=False):
    sub = sub.reset_index(drop=True)
    gsub = make_spatial_groups(sub, cfg)

    # group 名字
    key_tuple = key if isinstance(key, tuple) else (key,)
    key_str = "_".join(safe_key(k) for k in key_tuple)

    for m in models:
        res = fit_oof_with_spatialcv(sub, cfg, gsub, m, cv)

        title = (
            f"{key} | {m}\n"
            f"R²={res.metrics['r2']:.3f}, "
            f"RMSE={res.metrics['rmse']:.3f}, "
            f"MAE={res.metrics['mae']:.3f}"
            )
        target_label = cfg["data"]["columns"]["target_bcf"]
        # 1) 观测-预测图
        obs_path = FIG / f"obs_pred_{key_str}_{m}.png"
        obs_pred_scatter(res.y_true, res.oof_pred, title, str(obs_path), target_label=target_label)

        # 2) 残差图（新增：Pred-Obs）
        resid_path = FIG / f"resid_{key_str}_{m}.png"
        residual_plot(
            res.y_true,
            res.oof_pred,
            title=title.replace("obs-pred", "residual"),
            out_png=str(resid_path),
            target_label=target_label
        )

        rows.append({"group": str(key_tuple), "model": m, **res.metrics})

metrics = pd.DataFrame(rows).sort_values(['group', 'model'])
metrics.to_excel(OUT / 'metrics_spatialcv.xlsx', index=False)
metrics.head(20)


,group,model,rmse,mae,r2
2,"('corn', 'acid')",ann,0.045689,0.034567,-2.036361
0,"('corn', 'acid')",rf,0.022720,0.016184,0.249143
3,"('corn', 'acid')",svm,0.026467,0.020023,-0.018955
1,"('corn', 'acid')",xgb,0.022861,0.016353,0.239773
6,"('corn', 'alkaline')",ann,0.086740,0.066360,-21.700387
4,"('corn', 'alkaline')",rf,0.016985,0.012304,0.129577
7,"('corn', 'alkaline')",svm,0.019357,0.014436,-0.130561
5,"('corn', 'alkaline')",xgb,0.016855,0.012093,0.142860
10,"('corn', 'neutral')",ann,0.074975,0.058973,-9.177186
8,"('corn', 'neutral')",rf,0.020552,0.015149,0.235276
